<a href="https://colab.research.google.com/github/Samriddhacoderho/Face_Mask_Detection_CNN/blob/main/face_mask_detection_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from torchvision import datasets, transforms

In [2]:
!pip install opendatasets
import opendatasets as od
od.download('https://www.kaggle.com/datasets/omkargurav/face-mask-dataset')

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: samriddharajsatyal
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/omkargurav/face-mask-dataset


100%|██████████| 163M/163M [00:01<00:00, 93.7MB/s]


In [3]:
dataset=datasets.ImageFolder('/content/face-mask-dataset/data')

In [4]:
dataset.class_to_idx

{'with_mask': 0, 'without_mask': 1}

In [5]:
#Transformations, Train-Test, DataSet
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset, DataLoader

train_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])


idx = np.arange(len(dataset))

train_idx, temp_idx = train_test_split(
    idx, test_size=0.2,
    stratify=dataset.targets,
    random_state=42
)

val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5,
    stratify=np.array(dataset.targets)[temp_idx],
    random_state=42
)

train_dataset = Subset(
    datasets.ImageFolder('/content/face-mask-dataset/data', transform=train_transform),
    train_idx
)

val_dataset = Subset(
    datasets.ImageFolder('/content/face-mask-dataset/data', transform=test_transform),
    val_idx
)

test_dataset = Subset(
    datasets.ImageFolder('/content/face-mask-dataset/data', transform=test_transform),
    test_idx
)

In [6]:
#Data Loaders
train_loader=DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader=DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader=DataLoader(test_dataset, batch_size=32, shuffle=False)

In [7]:
img,lbl=next(iter(train_loader))
img

tensor([[[[-0.2684, -0.4054, -1.0219,  ..., -0.4397, -0.0629, -0.2856],
          [-0.2684, -0.4054, -1.0219,  ..., -0.4397, -0.0629, -0.2856],
          [-0.2513, -0.2342,  0.2624,  ...,  0.5193, -0.0801, -0.2684],
          ...,
          [-0.2684, -0.0801,  1.2214,  ...,  1.2043, -0.0972, -0.2684],
          [-0.2684, -0.0801,  1.2214,  ...,  1.2214, -0.0972, -0.2684],
          [-0.2684, -0.0801,  1.2214,  ...,  1.2385, -0.0972, -0.2684]],

         [[ 1.1506,  0.8704, -0.7927,  ..., -0.1800,  1.2206,  1.1331],
          [ 1.1506,  0.8704, -0.7752,  ..., -0.1800,  1.2206,  1.1331],
          [ 1.1506,  1.0630,  0.5553,  ...,  0.8004,  1.2031,  1.1506],
          ...,
          [ 1.1506,  1.1856,  1.5357,  ...,  1.5357,  1.1681,  1.1506],
          [ 1.1506,  1.1856,  1.5357,  ...,  1.5182,  1.1681,  1.1506],
          [ 1.1506,  1.1856,  1.5357,  ...,  1.5182,  1.1681,  1.1506]],

         [[ 2.2043,  1.8557, -0.4624,  ...,  0.1128,  2.2043,  2.2217],
          [ 2.2043,  1.8557, -

In [14]:
#model creation

import torch.nn as nn
from torchvision import models

model=models.mobilenet_v2(weights='DEFAULT')

for param in model.parameters():
  param.requires_grad=False

num_ftrs=model.classifier[1].in_features

model.classifier=nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(num_ftrs, 128),
    nn.ReLU(),
    nn.Linear(128, 2) # 2 classes: Mask, No Mask
)



In [11]:
print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)


In [13]:
print(model.classifier[1])

Linear(in_features=1280, out_features=1000, bias=True)


In [15]:
import torch.optim as optim

criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.classifier.parameters(),lr=0.001)